# VehicleTrajectNet — Demo Walkthrough

**This notebook is the live demo for the Honda interview.**

It walks through the full pipeline end-to-end:
1. Load a trained model from checkpoint
2. Run inference on one trajectory sample
3. Compare single-mode vs multi-modal predictions visually
4. Export an animated GIF
5. Show the quantitative results table

Every cell is self-contained and runs in order without any manual setup —
just `Kernel → Restart & Run All` before the interview.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from IPython.display import Image, display

from dataset   import TrajectoryDataset, PAST_STEPS, FUTURE_STEPS
from inference import load_model, predict_one, is_multimodal
from evaluate  import compute_ade, compute_fde, compute_min_ade, compute_min_fde
from visualise import export_prediction_gif

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print('All imports OK')

## 1 — Load models from checkpoint

In [ ]:
# Adjust paths if your checkpoints are named differently
BASELINE_CKPT  = '../checkpoints/baseline_best.pt'
MULTIMODAL_CKPT = '../checkpoints/best.pt'   # most recent best
PARQUET_PATH   = '../data/trajectories.parquet'

print('Loading baseline model...')
model_baseline = load_model(BASELINE_CKPT, device=DEVICE)

print('\nLoading multi-modal model...')
model_multi = load_model(MULTIMODAL_CKPT, device=DEVICE)

## 2 — Load validation dataset

In [ ]:
val_ds = TrajectoryDataset(PARQUET_PATH, split='val', augment_rotation=False)
print(f'Validation set: {len(val_ds)} trajectory windows')

## 3 — Single sample inference

Pick one sample and run both models on it.

In [ ]:
SAMPLE_IDX = 0  # change this to explore different samples

past_seq, future_seq = val_ds[SAMPLE_IDX]

# Run both models
pred_baseline = predict_one(model_baseline, past_seq, device=DEVICE)  # (6, 2)
pred_multi    = predict_one(model_multi,    past_seq, device=DEVICE)  # (K, 6, 2)

print(f'Past sequence shape   : {past_seq.shape}')   # (8, 4)
print(f'Future (ground truth) : {future_seq.shape}')  # (6, 2)
print(f'Baseline prediction   : {pred_baseline.shape}')  # (6, 2)
print(f'Multi-modal prediction: {pred_multi.shape}')      # (K, 6, 2)

# Per-sample metrics
gt = future_seq.unsqueeze(0)  # (1, 6, 2)
pb = pred_baseline.unsqueeze(0)          # (1, 6, 2)
pm = pred_multi.unsqueeze(0)             # (1, K, 6, 2)

print(f'\nBaseline  ADE : {compute_ade(pb, gt).item():.3f} m')
print(f'Baseline  FDE : {compute_fde(pb, gt).item():.3f} m')
print(f'minADE@K      : {compute_min_ade(pm, gt).item():.3f} m')
print(f'minFDE@K      : {compute_min_fde(pm, gt).item():.3f} m')

## 4 — Visualise predictions

Static plot comparing baseline vs multi-modal on the same sample.

In [ ]:
def plot_predictions(past_seq, future_seq, pred_baseline, pred_multi, sample_idx=0):
    """
    Side-by-side: baseline (single future) vs multi-modal (K futures).
    All coordinates are agent-centric — agent is at (0,0) facing right.
    """
    past_xy   = past_seq[:, :2].numpy()
    future_gt = future_seq.numpy()
    pb        = pred_baseline.cpu().numpy()  # (6, 2)
    pm        = pred_multi.cpu().numpy()     # (K, 6, 2)
    K         = pm.shape[0]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor='#0d1117')
    fig.suptitle(f'Sample {sample_idx} — Agent-centric frame (origin = current position)',
                 color='#c9d1d9', fontsize=11)

    for ax, title, preds, multimodal in [
        (axes[0], 'Baseline (single mode)', pb[np.newaxis], False),
        (axes[1], f'Multi-modal (K={K} modes)', pm, True),
    ]:
        ax.set_facecolor('#0d1117')
        ax.set_title(title, color='#c9d1d9', fontsize=10)
        ax.tick_params(colors='#484f58', labelsize=8)
        for spine in ax.spines.values():
            spine.set_edgecolor('#21262d')
        ax.grid(True, color='#161b22', linewidth=0.5)
        ax.set_aspect('equal')

        # Past track
        ax.plot(past_xy[:, 0], past_xy[:, 1],
                color='#8b949e', linewidth=2.0, label='Past track', zorder=3)
        ax.scatter(*past_xy[-1], color='white', s=60, zorder=6, label='Current position')

        # Predictions
        alpha_vals = np.linspace(0.9, 0.35, K) if multimodal else [0.9]
        for k in range(len(preds)):
            traj = np.vstack([past_xy[-1:], preds[k]])
            ax.plot(traj[:, 0], traj[:, 1],
                    color='#388bfd', linewidth=2.0, alpha=alpha_vals[k],
                    label='Prediction' if k == 0 else '_', zorder=4)
            ax.scatter(*preds[k][-1], color='#388bfd', s=40, alpha=alpha_vals[k], zorder=4)

        # Ground truth
        gt_traj = np.vstack([past_xy[-1:], future_gt])
        ax.plot(gt_traj[:, 0], gt_traj[:, 1],
                color='#3fb950', linewidth=2.2, label='Ground truth', zorder=5)
        ax.scatter(*future_gt[-1], color='#3fb950', s=50, zorder=5)

        ax.legend(facecolor='#161b22', edgecolor='#21262d',
                  labelcolor='#c9d1d9', fontsize=8, loc='upper left')

    plt.tight_layout()
    Path('../outputs').mkdir(exist_ok=True)
    save_path = f'../outputs/demo_sample_{sample_idx}.png'
    plt.savefig(save_path, dpi=120, bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    print(f'Saved: {save_path}')


plot_predictions(past_seq, future_seq, pred_baseline, pred_multi, SAMPLE_IDX)

## 5 — Export animated GIF

In [ ]:
gif_path = f'../outputs/demo_multimodal_{SAMPLE_IDX}.gif'

export_prediction_gif(
    model       = model_multi,
    dataset     = val_ds,
    sample_idx  = SAMPLE_IDX,
    output_path = gif_path,
    multimodal  = True,
    fps         = 4,
    device      = DEVICE,
)

# Display inline in notebook
display(Image(gif_path))

## 6 — Full validation set metrics

Compute ADE/FDE over the entire validation split.

In [ ]:
from torch.utils.data import DataLoader

val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)

@torch.no_grad()
def run_full_eval(model, loader, multimodal=False):
    model.eval()
    total_ade = total_fde = n = 0
    for past, future in loader:
        past, future = past.to(DEVICE), future.to(DEVICE)
        pred = model(past)
        if multimodal:
            ade = compute_min_ade(pred, future)
            fde = compute_min_fde(pred, future)
        else:
            ade = compute_ade(pred, future)
            fde = compute_fde(pred, future)
        b = past.size(0)
        total_ade += ade.item() * b
        total_fde += fde.item() * b
        n         += b
    return total_ade / n, total_fde / n

print('Evaluating baseline model...')
b_ade, b_fde = run_full_eval(model_baseline, val_loader, multimodal=False)

print('Evaluating multi-modal model...')
m_ade, m_fde = run_full_eval(model_multi, val_loader, multimodal=True)

K = getattr(model_multi, 'K', 5)

print(f'\n{"="*50}')
print(f'{"Model":<30} {"ADE (m)":>8} {"FDE (m)":>8}')
print(f'{"─"*50}')
print(f'{"Baseline (single-mode)":<30} {b_ade:>8.3f} {b_fde:>8.3f}')
print(f'{f"Multi-modal (minADE@{K})":<30} {m_ade:>8.3f} {m_fde:>8.3f}')
print(f'{"─"*50}')
improvement = (b_ade - m_ade) / b_ade * 100
print(f'ADE improvement from K={K}: {improvement:+.1f}%')
print(f'{"="*50}')

## 7 — Explore multiple samples interactively

Use this cell to browse different validation samples during the interview.

In [ ]:
# Change this index to explore different scenarios
# Interesting ones to look for:
#   - Samples where multi-modal clearly fans out at an intersection
#   - Samples where baseline predicts straight but GT turns
#   - Samples where one of the K modes perfectly matches GT

EXPLORE_IDX = 5  # <-- change me

past_ex, future_ex = val_ds[EXPLORE_IDX]
pred_b_ex = predict_one(model_baseline, past_ex, device=DEVICE)
pred_m_ex = predict_one(model_multi,    past_ex, device=DEVICE)

plot_predictions(past_ex, future_ex, pred_b_ex, pred_m_ex, EXPLORE_IDX)

## Architecture summary

Quick reference for the interview — no need to memorise, just understand each bullet.

**Input:** Past 4 seconds → 8 timesteps × 4 features `[x, y, heading, velocity]`, agent-centric normalised.

**Encoder:** 2-layer LSTM, hidden=128. Context vector = `cat(h_n, c_n)` — both hidden and cell state, so the decoder sees short-term *and* long-term memory.

**Decoder (baseline):** MLP 256→64→12, reshaped to `(6, 2)` — one predicted future.

**Decoder (multi-modal):** MLP 256→64→60, reshaped to `(5, 6, 2)` — K=5 candidate futures, trained with best-of-K loss.

**Why agent-centric?** Raw map coordinates don't generalise — a car at `(400,500)` going east looks different from one at `(100,200)` going east. Normalising to origin forces the model to learn physics, not geography.

**Why multi-modal?** At an intersection there are 3 possible futures. A single-mode model is forced to predict their average — which is often off the road. K=5 modes let the model hedge.

**Improvements over naive baseline:**
- `cat(h_n, c_n)` context (not just `h_n`)
- Angle wrapping for heading features
- Velocity normalisation
- Rotation augmentation during training